# 🎓 WE4 · Notebook 04: PPO
## Teaching an agent to play Flappy Bird

This notebook builds up to PPO the way the lecture did: play the game, try the
simplest policy gradient there is, watch it get stuck, and then fix it.

**Wherever you see 🎯 there is something for you to write.** Four tasks, nine
short lines in all: the loop that plays one life, REINFORCE's objective, the
critic network, and the three lines of PPO's objective. Everything else is
already here, and every task is followed by a check that runs in about a second
and names the mistake.

By the end you will have three clips of the same bird: untrained, taught by
REINFORCE, and taught by PPO.

### Before you start

**Runtime, then Change runtime type, then choose CPU.** Not GPU.

The networks here are tiny (12 inputs, two hidden layers of 64) and the agent
plays one step at a time, so a GPU only adds launch latency to every one of a
hundred thousand tiny calls. It would be slower, and it would spend your GPU
quota.

### The plan

| | |
|---|---|
| 1 | Set up. Four cells to run, nothing to read |
| 2 | Meet the environment, and 🎯 play one whole life yourself |
| 3 | Meet the actor, and watch an untrained one fail |
| 4 | 🎯 Write REINFORCE's objective, train for 40 seconds, and watch it get stuck |
| 5 | 🎯 Write the critic, the thing REINFORCE was missing |
| 6 | 🎯 Write the three lines of PPO's objective, and check them in one second |
| 7 | Train for about two minutes |
| 8 | Watch all three side by side |

## 1. Setup

The next four cells are plumbing: installing, importing, and defining helpers to
turn frames into video. **Run them and move on.** Nothing in them is part of the
exercise, and none of it is examinable. Click "Show code" if you are curious.

In [ ]:
#@title Run me: install
# This installs almost nothing: Colab already ships gymnasium, pygame and torch.
# The pins are here so the notebook still works if Google changes the image.
# Do not add -q: it would hide a "RESTART SESSION" banner if that ever happens.
!pip install "gymnasium==1.3.0" "flappy-bird-gymnasium==0.4.0"

In [ ]:
#@title Run me: version check
# Version guard. If anything goes wrong later, the output of this cell is the
# first thing to look at.
import gymnasium, numpy, torch, flappy_bird_gymnasium
print("gymnasium", gymnasium.__version__)
print("numpy    ", numpy.__version__)
print("torch    ", torch.__version__)
assert gymnasium.__version__.startswith("1."), "expected gymnasium 1.x"
print("\nOK")

In [ ]:
#@title Run me: imports, and one frame of the game
import numpy as np, torch, torch.nn as nn, time
import gymnasium as gym
import flappy_bird_gymnasium   # registers FlappyBird-v0

torch.set_num_threads(1)       # measured: no slower, and it keeps runs reproducible

def make_env(render=False):
    return gym.make("FlappyBird-v0",
                    render_mode="rgb_array" if render else None,
                    use_lidar=False)

env = make_env(render=True)
print("state :", env.observation_space)
print("action:", env.action_space)

obs, _ = env.reset(seed=0)
frame = env.render()
print("one rendered frame:", frame.shape)
env.close()

import matplotlib.pyplot as plt
plt.figure(figsize=(3, 5)); plt.imshow(frame); plt.axis("off"); plt.show()

In [ ]:
#@title Run me: a helper that turns frames into video
from base64 import b64encode
from IPython.display import HTML
import imageio

def save_video(frames, path, fps=30):
    '''Write frames to mp4, falling back to gif if ffmpeg is unavailable.'''
    try:
        imageio.mimsave(path, frames, fps=fps, macro_block_size=1)
        return path
    except Exception as e:
        print("mp4 failed (%s), falling back to gif" % type(e).__name__)
        gif = path.replace(".mp4", ".gif")
        imageio.mimsave(gif, frames[::2], duration=1000 / (fps / 2), loop=0)
        return gif

def show_video(path, width=260, caption=""):
    data = b64encode(open(path, "rb").read()).decode()
    if path.endswith(".gif"):
        tag = '<img width="%d" src="data:image/gif;base64,%s">' % (width, data)
    else:
        tag = ('<video width="%d" autoplay loop controls>'
               '<source src="data:video/mp4;base64,%s" type="video/mp4"></video>') % (width, data)
    return '<figure style="margin:0 12px 0 0;text-align:center">%s<figcaption style="font:13px sans-serif;color:#555">%s</figcaption></figure>' % (tag, caption)

def play(*figs):
    display(HTML('<div style="display:flex;align-items:flex-start">%s</div>' % "".join(figs)))

## 2. The environment

Everything the lecture defined has a concrete meaning here.

| Lecture | Flappy Bird |
|---|---|
| **state** | 12 numbers: the bird's height and speed, and the positions of the next pipes |
| **action** | 2 choices: `0` do nothing, `1` flap |
| **reward** | `+0.1` for staying alive one frame, `+1.0` for passing a pipe, `-1.0` for dying, and `-0.5` for every frame spent off the top of the screen |
| **episode** | one life, from the start until the bird hits a pipe or the ground |
| **return** | everything collected in one life |

### How you talk to it

Two methods do everything.

`env.reset()` starts a new life and hands back the first state.

`env.step(action)` takes one action and hands back **five** things:

| what comes back | what it means | in Flappy Bird |
|---|---|---|
| `observation` | the new state, after your action | the 12 numbers again, updated |
| `reward` | what that single action earned | `+0.1`, or `+1.0` at a pipe, or `-0.5` off the top, or `-1.0` if it just died |
| `terminated` | did the episode end **because of the task** | the bird crashed |
| `truncated` | was the episode cut short **from outside**, e.g. a time limit | never happens in this game |
| `info` | extra diagnostics, ignored by the algorithm | the score so far |

`terminated` and `truncated` are separate because they mean different things to
the algorithm. When an episode *terminates* there is genuinely no future left to
value. When it is merely *truncated* the future still exists, it just stopped
being recorded. This game only ever terminates, so the code below treats them
together.

### The five methods you need

| method | what it does | what it gives back |
|---|---|---|
| `env.reset(seed=...)` | start a new life | `(observation, info)`, a pair |
| `env.step(action)` | take one action | the five things above, in that order |
| `env.action_space.sample()` | draw a legal action at random | an action, here `0` or `1` |
| `env.render()` | hand back the current frame as pixels | an array of shape `(512, 288, 3)` |
| `env.close()` | let the game go | nothing |

The whole of reinforcement learning is a loop over `reset` and `step`. The cell
below calls each of the first two once, so you can see exactly what comes back.

In [ ]:
env = make_env()

obs, info = env.reset(seed=0)
print("env.reset() gives back a PAIR:")
print("   observation:", np.round(obs, 3))
print("   info       :", info)

obs, reward, terminated, truncated, info = env.step(1)      # 1 = flap
print("\nenv.step(1) gives back FIVE things, in this order:")
print("   observation:", np.round(obs, 3))
print("   reward     :", reward)
print("   terminated :", terminated)
print("   truncated  :", truncated)
print("   info       :", info)

print("\nand a legal random action, for when there is no policy yet:")
print("   env.action_space.sample() ->", env.action_space.sample())
env.close()

### 🎯 Task 1: one whole life, played at random

The loop below is the shape of every reinforcement learning program ever written:
act, observe, add up the reward, stop when the episode ends. There are no neural
networks in it and nothing of PPO. It is only the methods in the table above.

**Fill in the four lines marked 🎯.** Everything you need is in that table.

Two details worth knowing:

- The loop is a `for` over `max_steps` rather than a `while True`, so that an
  unfinished line 4 cannot hang your notebook. A correct loop breaks out long
  before the cap.
- `env.reset(seed=...)` seeds the *game*, but the action space draws its random
  actions from a separate generator, so it gets its own `env.action_space.seed(...)`.
  Seeding both is what makes everyone in the room see the same life.

In [ ]:
def play_one_life(env, seed=0, max_steps=2000):
    '''Play one whole life choosing actions at random.

    Returns (steps, total_reward): how many steps the bird survived, and the
    RETURN for that life, meaning every reward it collected added up.
    '''
    obs, info = env.reset(seed=seed)
    total_reward, steps = 0.0, 0

    for _ in range(max_steps):

        # 🎯 1. Pick an action at random. There is no policy yet, so this is a
        #       coin flip. The action space can hand you a legal one.
        action = None  # 🎯 replace this

        # 🎯 2. Take that action in the game. The five things it gives back are
        #       already unpacked for you, in the order the table above lists
        #       them, so you only need the call itself.
        obs, reward, terminated, truncated, info = None  # 🎯 replace this

        # 🎯 3. Add what this one step earned to the running total. This is the
        #       whole definition of the return: the rewards, added up.
        total_reward += None  # 🎯 replace this

        steps += 1

        # 🎯 4. Stop as soon as the life is over, whether the bird crashed or
        #       the episode was cut short from outside.
        if None:
            break

    return steps, total_reward

In [ ]:
#@title Run me: check your loop
class _Spy(gym.Wrapper):
    '''Watches every call your loop makes, so the check can be specific.'''
    def __init__(self, env):
        super().__init__(env)
        self.actions, self.rewards, self.dones = [], [], []
    def step(self, action):
        self.actions.append(action)
        out = self.env.step(action)
        self.rewards.append(float(out[1]))
        self.dones.append(bool(out[2] or out[3]))
        return out

def check_loop():
    spy = _Spy(make_env())
    spy.action_space.seed(0)
    try:
        steps, total = play_one_life(spy, seed=0, max_steps=300)
    except Exception as e:
        msg = str(e)
        if spy.actions and spy.actions[-1] is None:
            print("STEP 1 is still None, so there was no action to play.")
            print("  Ask the action space for a random legal one.")
        elif "unpack" in msg:
            print("STEP 2 is still None. Put the env.step(...) call on the right")
            print("  of the equals sign, so there are five things to unpack.")
        elif "+=" in msg or "unsupported operand" in msg:
            print("STEP 3 is still None. Add this step's reward to the total.")
        else:
            print("Your loop raised %s: %s" % (type(e).__name__, msg))
        spy.close(); return

    n = len(spy.actions)
    spy.close()

    if n == 0:
        print("FAIL: your loop never called env.step, so nothing was played."); return

    # STEP 1: were the actions random draws from the action space?
    if any(a is None for a in spy.actions):
        print("FAIL on STEP 1, the action.")
        print("  It is still None. The game quietly took that as 'do not flap',")
        print("  so nothing crashed. Ask the action space for a random legal one.")
        return
    ref = gym.spaces.Discrete(2); ref.seed(0)
    want = [int(ref.sample()) for _ in range(n)]
    taken = [int(a) for a in spy.actions]
    if taken != want:
        print("FAIL on STEP 1, the action.")
        if len(set(taken)) == 1:
            print("  Every action was %d. The bird needs to be flapping at random," % taken[0])
            print("  not doing the same thing forever. Ask the action space for one.")
        else:
            print("  The actions taken were not the draws the action space would give.")
        return

    # STEP 4: did it stop exactly when the life ended?
    if any(spy.dones[:-1]):
        print("FAIL on STEP 4, the stopping condition.")
        print("  You kept playing after the life had already ended.")
        print("  Both terminated and truncated mean it is over.")
        return
    if not spy.dones[-1]:
        print("FAIL on STEP 4, the stopping condition.")
        print("  Your loop ran to the %d step cap without ever breaking out," % n)
        print("  so the condition never became true. It should end the life.")
        return

    # STEP 3: is total_reward really the sum of the rewards?
    if steps != n:
        print("FAIL: steps came out as %s but env.step was called %d times." % (steps, n)); return
    want_total = sum(spy.rewards)
    if abs(total - want_total) > 1e-6:
        print("FAIL on STEP 3, the running total.")
        print("  you got %.4f, the rewards actually add up to %.4f" % (total, want_total))
        if abs(total) < 1e-9:
            print("  Nothing was ever added: the total is still its starting 0.0.")
        elif abs(total - n) < 1e-6:
            print("  You added 1 per step, which counts steps. Add what the step EARNED.")
        return

    # STEP 4 again, on an env that IS cut short from outside, so that a
    # condition testing only terminated cannot slip through.
    short = _Spy(gym.wrappers.TimeLimit(make_env(), max_episode_steps=20))
    short.action_space.seed(0)
    try:
        s2, _ = play_one_life(short, seed=0, max_steps=300)
    finally:
        short.close()
    if s2 != 20:
        print("FAIL on STEP 4, the stopping condition.")
        print("  On a life cut short from outside after 20 steps, your loop ran for")
        print("  %d. truncated ends the life exactly as terminated does, and the" % s2)
        print("  condition has to accept either one.")
        return

    print("PASS. Your loop plays a whole life and returns %d steps, return %.2f." % (steps, total))
    print("      That return is one sample of exactly the number PPO is built to raise.")

check_loop()

Now run it for real. The cap is generous here; a random agent never gets near it.

In [ ]:
env = make_env()
env.action_space.seed(0)          # so everyone's coin flips match

steps, total_reward = play_one_life(env, seed=0)
env.close()

print("a random agent survived %d steps (%.1f seconds)" % (steps, steps / 30))
print("its RETURN for that life was %.2f" % total_reward)
print("\nthat return is the number the whole algorithm is trying to make bigger.")

Negative, and over in under two seconds. Flapping at random means flapping about
half the time, which drives the bird off the top of the screen and holds it
there at `-0.5` a frame, so the return comes out well below what merely
surviving those frames would have paid. Everything from here is about turning
that number positive.

## 3. The actor, and an untrained one

The **actor** is the policy. It answers *"how much do I like each action here?"*,
so a state goes in and **one number per action** comes out. That is the only
network needed for the next section; a second one arrives when we find out what
this one cannot do on its own.

One line in the next cell is worth reading slowly, because it is how a network
turns into a decision:

```python
a = int(torch.argmax(actor(torch.as_tensor(obs, dtype=torch.float32))))
```

Inside out:

1. `torch.as_tensor(obs, dtype=torch.float32)` turns the 12 numbers into a tensor
   the network can accept. The cast matters: the weights are float32 and torch
   will not quietly mix types.
2. `actor(...)` pushes them through the network. Out come **two numbers, one per
   action**. They are *logits*, unbounded scores, not probabilities. Higher means
   the policy likes that action more.
3. `torch.argmax(...)` takes the index of the larger one, so `0` or `1`.
4. `int(...)` unwraps it to a plain integer, which is what `env.step` wants.

It is wrapped in `with torch.no_grad():`, which tells torch **not to record any
of this for training**. While playing we only want the answer, not the machinery
for differentiating it. That is also why the log-probabilities stored during
play are frozen constants later: they were computed with the gradient turned off.

Nothing to write here. Run it, and watch an actor that has never learned
anything.

In [ ]:
class Actor(nn.Module):
    '''Given a state, how much do I like each action?'''

    def __init__(self, n_obs=12, n_act=2):
        super().__init__()
        # n_obs numbers in, two hidden layers of 64 with tanh, and n_act numbers
        # out: one score per action.
        self.net = nn.Sequential(nn.Linear(n_obs, 64), nn.Tanh(),
                                 nn.Linear(64, 64),    nn.Tanh(),
                                 nn.Linear(64, n_act))

    def forward(self, obs):
        return self.net(obs)                    # logits, one per action

    def dist(self, obs):
        '''The policy itself: those scores, turned into a choice we can sample.'''
        return torch.distributions.Categorical(logits=self.net(obs))


def rollout(actor, seed=0, max_frames=1200):
    '''Play one life with the actor's best-guess action and record the frames.'''
    env = make_env(render=True)
    obs, _ = env.reset(seed=seed)
    frames, total, pipes = [], 0.0, 0
    for _ in range(max_frames):
        frames.append(env.render())
        with torch.no_grad():
            a = int(torch.argmax(actor(torch.as_tensor(obs, dtype=torch.float32))))
        obs, r, term, trunc, _ = env.step(a)
        total += float(r)
        if r >= 1.0: pipes += 1
        if term or trunc: break
    env.close()
    return frames, total, pipes

In [ ]:
SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)

untrained_actor = Actor()
frames_before, ret_before, pipes_before = rollout(untrained_actor, seed=SEED)
print("untrained: survived %d frames (%.1f seconds), %d pipes, return %.2f"
      % (len(frames_before), len(frames_before) / 30, pipes_before, ret_before))

before_path = save_video(frames_before, "before.mp4")
play(show_video(before_path, caption="untrained: return %.2f" % ret_before))

It flaps on every single frame, pins itself against the top of the screen, and
dies up there. Its return is about `-9`, which is **worse** than the coin-flipping
loop you wrote in Task 1. An untrained network is not neutral or cautious: it has
an arbitrary opinion and follows it without wavering.

## 4. 🎯 Task 2: REINFORCE

REINFORCE is the simplest policy gradient there is, and it fits on one line.

Play some complete lives. For every action taken, work out $G_t$, **the return
from that step to the end of that life**. Then push up the log-probability of the
actions that were followed by a good return, and push down the ones that were
not:

$$\nabla J(\theta) \;=\; \mathbb{E}\big[\,G_t \, \nabla \log \pi_\theta(a_t \mid s_t)\,\big]$$

No value function, no probability ratio, no clipping. Just: whatever preceded a
good outcome, do more of it.

Handed to an optimiser, that is a product, an average, and a minus sign.

In [ ]:
def reinforce_loss(logp, G):
    '''
    logp : log-probability the policy gave each action it actually took
    G    : the return from that step to the end of its life, already
           standardised across the batch

    Returns one number, to be minimised.
    '''

    # 🎯 Push up the log-probability of the actions that were followed by a good
    #    return, and down the ones that were not. Multiply the two, average over
    #    the batch, and negate it so that minimising raises the return.
    loss = None  # 🎯 replace this

    return loss

### Check your line before training anything

In [ ]:
#@title Run me: check your REINFORCE line
def check_reinforce():
    logp = torch.tensor([-0.30, -1.20, -0.05, -2.00, -0.70])
    G    = torch.tensor([ 1.50, -0.80,  2.00,  0.60, -1.30])

    try:
        loss = reinforce_loss(logp, G)
    except Exception as e:
        print("Your code raised %s: %s" % (type(e).__name__, e)); return

    if loss is None:
        print("FAIL: loss is still None. Write the line."); return

    if not torch.is_tensor(loss):
        print("FAIL: the loss has to be a torch tensor, but you returned a %s."
              % type(loss).__name__)
        print("      Build it out of logp and G, so that torch can differentiate it.")
        return

    if loss.dim() != 0:
        print("FAIL: the loss has to be a single number, but you returned shape %s."
              % (tuple(loss.shape),))
        print("      One number per step is not a loss. Average over the batch.")
        return

    v = float(loss)
    if abs(v - (-0.024)) < 1e-5:
        print("PASS. Go and train.")
        print("(here logp * G is [-0.45, 0.96, -0.10, -1.20, 0.91], which averages to 0.024)")
        return

    print("Not right yet: you got %.6f, expected -0.024000" % v)
    if abs(v - 0.024) < 1e-5:
        print("  Right number, wrong sign. We want the return to go UP and an")
        print("  optimiser only goes DOWN, so the loss is the negative.")
    elif abs(v - (-0.12)) < 1e-5:
        print("  That is the sum rather than the average. Use .mean().")
    elif abs(v - 0.12) < 1e-5:
        print("  That is the sum, and the wrong sign.")
    elif abs(v - 0.85) < 1e-5:
        print("  That is the log-probabilities on their own. G is not being used,")
        print("  so this would push up every action taken, good or bad.")
    else:
        print("  Expected: -(logp * G).mean()")

check_reinforce()

### Train it

**Nothing below is marked 🎯: you do not write any of it.** It arrives in three
small pieces so that none of them is a wall, and the last one runs for about
**40 seconds**.

Here is the whole thing in pseudocode first:

```
until 102,400 steps of the game have been played:

  1. PLAY whole lives, until at least 2048 steps are in hand.
     WHOLE lives, because REINFORCE cannot score an action until it has
     seen how the life it belonged to turned out.

  2. SCORE
     for each life, walk BACKWARDS through it:
        G(t) = reward(t) + discount x G(t+1)
     then standardise G across the batch

  3. IMPROVE
     ONE gradient step:
        loss = YOUR ONE LINE
             - 0.01 x (actor's entropy)      # pay it to stay undecided

  throw the batch away
```

**One** gradient step per 2048 steps of play. Remember that number: PPO's will be
320, off the same data.

#### First, the scoring

$G_t$ is the return from step $t$ to the end of that life. Adding it up forwards
would mean a fresh sum for every step. Walking **backwards** gets all of them in
one pass, because each step's answer is just its own reward plus the discounted
answer of the step after it:

$$G_t \;=\; r_t + \gamma\, G_{t+1}$$

In [ ]:
GAMMA, ENT_COEF = 0.99, 0.01                     # both shared with PPO later
RF_STEPS, RF_BATCH, RF_LR = 102_400, 2048, 1e-3

def returns_to_go(rewards, gamma=GAMMA):
    '''G for every step of one life: what the rest of that life was worth.'''
    g, out = 0.0, []
    for r in reversed(rewards):
        g = r + gamma * g
        out.append(g)
    return out[::-1]                             # back into playing order

#### Then the playing, which you have already written

The cell below is the loop from **Task 1**. Three things changed, and they are
the three lines marked `# <-- new`: the coin flip became the actor, what happened
gets written down instead of thrown away, and it keeps starting fresh lives until
the batch is full.

In [ ]:
def collect_lives(actor, env, min_steps):
    '''Play whole lives until at least min_steps have been collected.'''
    O, A, G, ep_returns = [], [], [], []

    while len(A) < min_steps:                          # <-- new: keep going
        obs, _ = env.reset()
        ep_o, ep_a, ep_r, done = [], [], [], False

        while not done:
            ot = torch.as_tensor(obs, dtype=torch.float32)
            with torch.no_grad():
                a = int(actor.dist(ot).sample())       # <-- was a coin flip
            obs, r, term, trunc, _ = env.step(a)
            ep_o.append(ot); ep_a.append(a); ep_r.append(float(r))   # <-- new
            done = term or trunc

        ep_returns.append(sum(ep_r))
        O += ep_o; A += ep_a
        G += returns_to_go(ep_r)                       # score the life just played

    return (torch.stack(O), torch.tensor(A),
            torch.tensor(G, dtype=torch.float32), ep_returns)

Run it once with the untrained actor, just to see what comes back. Nothing is
learned here; this is only to make the two cells above concrete before they
disappear inside a training loop.

In [ ]:
env = make_env()
env.reset(seed=SEED)
O, A, G, ep_returns = collect_lives(untrained_actor, env, RF_BATCH)
env.close()

print("states :", tuple(O.shape), " (one row of 12 numbers per step)")
print("actions:", tuple(A.shape))
print("returns:", tuple(G.shape))
print("lives  : %d of them, averaging %.2f" % (len(ep_returns), np.mean(ep_returns)))

print("\nG for the last 6 steps of the batch:", [round(float(x), 2) for x in G[-6:]])
print("the last one is just that step's own reward: the life ended there,")
print("so there was no future left to add.")

#### And the loop that uses them

This is the pseudocode above, as code.

In [ ]:
def train_reinforce(seed=SEED, total_steps=RF_STEPS, batch=RF_BATCH, lr=RF_LR):
    torch.manual_seed(seed); np.random.seed(seed)
    env = make_env()
    env.reset(seed=seed)                          # seed the game too, so re-running repeats
    actor = Actor(env.observation_space.shape[0], env.action_space.n)
    opt = torch.optim.Adam(actor.parameters(), lr=lr)
    log, xs, curve, seen, t0 = [], [], [], 0, time.time()

    while seen < total_steps:
        O, A, G, ep_returns = collect_lives(actor, env, batch)   # 1. PLAY
        seen += len(A); log += ep_returns

        G = (G - G.mean()) / (G.std() + 1e-8)                    # 2. SCORE

        d = actor.dist(O)                                        # 3. IMPROVE, once
        loss = reinforce_loss(d.log_prob(A), G) - ENT_COEF * d.entropy().mean()
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(actor.parameters(), 0.5)
        opt.step()

        xs.append(seen); curve.append(float(np.mean(log[-20:])))
        if len(curve) % 10 == 0:
            print("steps %6d   mean return %6.2f   %4.0fs"
                  % (seen, curve[-1], time.time() - t0), flush=True)
    env.close()
    return actor, xs, curve

reinforce_actor, rf_xs, rf_curve = train_reinforce()

In [ ]:
frames_rf, ret_rf, pipes_rf = rollout(reinforce_actor, seed=SEED)
print("REINFORCE: survived %d frames (%.1f seconds), %d pipes, return %.2f"
      % (len(frames_rf), len(frames_rf) / 30, pipes_rf, ret_rf))
print("untrained: survived %d frames (%.1f seconds), %d pipes, return %.2f"
      % (len(frames_before), len(frames_before) / 30, pipes_before, ret_before))

rf_path = save_video(frames_rf, "reinforce.mp4")
play(show_video(before_path, caption="untrained: return %.2f" % ret_before),
     show_video(rf_path,     caption="after REINFORCE: return %.2f" % ret_rf))

## 5. What REINFORCE learned, and what it did not

The return went from about `-9` to about `+2`, and that is real: the untrained
bird flapped every frame and pinned itself to the ceiling, and this one has
stopped doing that. It glides down and lands.

It also stopped **there**. The curve climbs for roughly the first 30,000 steps
and is flat for the remaining 70,000, and the bird never reaches a pipe. Giving
it six times as long does not change that. This was measured, not assumed.

The reason is the whole point of the next two sections.

REINFORCE multiplies each action's log-probability by $G_t$, the return of the
rest of that life. That is **one blunt number shared by every action in the
life**. If the life went badly, every action in it is pushed down, including the
good ones. If it went well, every action is pushed up, including the reckless
ones. The direction is right on average and wrong on very nearly every
individual step, and the only remedy REINFORCE has is to play yet more lives.

So the policy finds the one move that reliably stops the bleeding, which is to
stop flapping, and from there every direction it can see looks worse than
standing still.

Two changes get it out of that rut, and PPO makes both:

| | what changes | what it needs |
|---|---|---|
| 1 | stop asking *"how did the whole life go"* and start asking *"was this action better than this state deserved"* | something that knows what a state is worth: the **critic**, Task 3 |
| 2 | stop throwing the batch away after one gradient step | the **probability ratio**, Task 4 |

### 🎯 Task 3: the critic

The **actor** answers *"how much do I like each action here?"*, so a state goes
in and **one number per action** comes out. You already have it:

```python
self.net = nn.Sequential(nn.Linear(n_obs, 64), nn.Tanh(),
                         nn.Linear(64, 64),    nn.Tanh(),
                         nn.Linear(64, n_act))
```

The **critic** answers *"how good is this state?"*, so a state goes in and
**exactly one number** comes out. One. Not one per action, because the critic
does not score actions at all: it scores the situation.

Write it. It is the same network right up to the last layer, and that last layer
is the entire conceptual difference between the two.

In [ ]:
class Critic(nn.Module):
    '''Given a state, how good is it?'''

    def __init__(self, n_obs=12):
        super().__init__()
        # 🎯 The same network as the Actor, except for how many numbers come out
        #    of the last layer. Note there is no n_act here at all: the critic
        #    never sees how many actions exist, because it does not score them.
        self.net = None  # 🎯 replace this

    def forward(self, obs):
        return self.net(obs).squeeze(-1)        # one value per state

In [ ]:
#@title Run me: check your critic
def check_critic():
    try:
        actor, critic = Actor(), Critic()
    except Exception as e:
        print("Building the networks raised %s: %s" % (type(e).__name__, e)); return

    if critic.net is None:
        print("FAIL: self.net is still None inside Critic. Write it, copying the Actor.")
        return

    try:
        states = torch.zeros(5, 12)          # five pretend states
        raw = critic.net(states)
        v = critic(states)
    except Exception as e:
        print("Your critic raised %s: %s" % (type(e).__name__, e))
        print("Check that the first layer accepts n_obs inputs.")
        return

    if tuple(raw.shape) != (5, 1):
        print("FAIL: given 5 states, your critic's last layer returned shape %s." % (tuple(raw.shape),))
        print("      It should return (5, 1): exactly one number per state.")
        if raw.shape[-1] == 2:
            print("      You gave it 2 outputs, one per action. The critic does not score")
            print("      actions, it scores the STATE, with one number however many")
            print("      actions there happen to be.")
        return

    one = torch.zeros(12)
    print("PASS. Your critic turns a state into a single number.")
    print("      actor  on one state -> %s, one score per action" %
          [round(float(x), 4) for x in actor(one)])
    print("      critic on one state -> %.4f, one value, full stop" % float(critic(one)))

check_critic()

## 6. 🎯 Task 4: PPO's objective

The critic fixes *what* each action is scored against. This fixes *how many
times* each batch of experience can be used.

### Step 1: the probability ratio

The 2048 steps were played by the policy **as it was before this update**. As soon
as you improve the policy once, that data was collected by somebody who no longer
exists. The probability ratio is how you keep using it anyway:

$$\rho_t(\theta) \;=\; \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\mathrm{old}}}(a_t \mid s_t)}$$

It asks: **how much more, or less, often would the policy I am building now have
taken this action?**

- $\rho_t > 1$: the new policy favours this action more than the collector did.
  The sample is more representative of the new policy, so it counts for more.
- $\rho_t < 1$: the new policy has moved away from this action. The sample says
  less about the new policy, so it counts for less.
- $\rho_t = 1$: the two policies agree here, and nothing is reweighted.

You are given the logarithms, `new_logp` and `old_logp`. A ratio is the
exponential of the difference of logarithms.

### Step 2: the clipped ratio

Left alone, that ratio is unbounded, and one sample could drag the policy
anywhere. So cap it:

$$\bar{\rho}_t \;=\; \operatorname{clip}(\rho_t,\; 1-\epsilon,\; 1+\epsilon)$$

If it is too large, it becomes $1+\epsilon$. If it is too small, it becomes
$1-\epsilon$. In torch this is `torch.clamp(x, lo, hi)`.

### Step 3: keep the pessimistic one

Now there are two possible multipliers for the same sample: the honest one
$\rho_t \widehat{A}_t$, and the capped one $\bar{\rho}_t \widehat{A}_t$.
PPO **takes the smaller of the two**:

$$L \;=\; \operatorname{average}\Big[\min\big(\rho_t \widehat{A}_t,\;\; \bar{\rho}_t \widehat{A}_t\big)\Big]$$

Taking the **minimum** means always believing the less flattering of the two
estimates. That one word is what makes the clipping do the right thing:

| advantage positive, and the ratio has | the clipped term is | `min` keeps | effect |
|---|---|---|---|
| grown past $1+\epsilon$ | smaller | the cap | no further reward for pushing |
| fallen below $1-\epsilon$ | larger | the honest $\rho_t \widehat{A}_t$ | the gradient still flows, so it can come back |

So clipping stops a sample pushing **further away** from $\rho = 1$, but never
stops it coming **back**. With `max` instead of `min` you would get exactly the
wrong behaviour in both rows.

One last thing: we want to **maximise** $L$, but optimisers **minimise**, so
return the negative.

Fill in the three lines below.

In [ ]:
CLIP_EPS = 0.2

def ppo_losses(new_logp, old_logp, adv, clip_eps=CLIP_EPS):
    '''
    new_logp : log of the probability the CURRENT policy gives the action taken
    old_logp : log of the probability the policy that COLLECTED the data gave it
    adv      : the advantage estimate for each step

    Returns (ratio, policy_loss).
    '''

    # 🎯 STEP 1. The probability ratio.
    #            Hint: exp(log a - log b) = a / b
    ratio = None  # 🎯 replace this

    # 🎯 STEP 2. The same ratio, capped below at 1-clip_eps and above at 1+clip_eps.
    #            Hint: torch.clamp(x, lo, hi)
    clipped_ratio = None  # 🎯 replace this

    # 🎯 STEP 3. Two candidate multipliers, ratio*adv and clipped_ratio*adv.
    #            Keep the SMALLER of the two for each step, average over the
    #            batch, and negate so it can be minimised.
    #            Hint: -torch.min(A, B).mean()
    policy_loss = None  # 🎯 replace this

    return ratio, policy_loss

### Check your three lines, before training anything

This runs in about a second on fixed numbers, and it names the specific
mistake. Do not start the two-minute training run until it says PASS.

In [ ]:
#@title Run me: check your three lines
def check():
    new_logp = torch.tensor([-0.30, -1.20, -0.05, -2.00, -0.70])
    old_logp = torch.tensor([-0.50, -0.90, -0.05, -1.10, -1.40])
    adv      = torch.tensor([ 1.50, -0.80,  2.00,  0.60, -1.30])

    try:
        ratio, loss = ppo_losses(new_logp, old_logp, adv, 0.2)
    except Exception as e:
        print("Your code raised %s: %s" % (type(e).__name__, e)); return

    if ratio is None or loss is None:
        print("FAIL: something is still None. All three steps need replacing."); return

    want_ratio = torch.tensor([1.221403, 0.740818, 1.0, 0.40657, 2.013753])
    if not torch.allclose(ratio, want_ratio, atol=1e-4):
        print("FAIL on STEP 1, the ratio.")
        print("  you     :", [round(float(x), 4) for x in ratio])
        print("  expected:", [round(float(x), 4) for x in want_ratio])
        print("  It must be 1.0 wherever new_logp equals old_logp. Check the order")
        print("  of the subtraction: it is new minus old.")
        return

    v = float(loss)
    if abs(v - (-0.157213)) < 1e-4:
        print("PASS. All three steps are right. Go and train.")
        print("(for reference, the clipped ratios here are [1.2, 0.8, 1.0, 0.8, 1.2])")
        return

    print("STEP 1 is right, but the final loss is not.")
    print("  you got %.6f, expected -0.157213" % v)
    if abs(v - 0.157213) < 1e-4:
        print("  Right number, wrong sign. Negate it.")
    elif abs(v - (-0.43189)) < 1e-4:
        print("  That is max() instead of min(). PPO keeps the SMALLER of the two")
        print("  candidates, the pessimistic one.")
    elif abs(v - (-0.173103)) < 1e-4:
        print("  That is ratio*adv only: STEP 2 is not being used in STEP 3.")
    elif abs(v - (-0.416)) < 1e-3:
        print("  That is clipped_ratio*adv only: you need BOTH candidates.")
    else:
        print("  Expected: -torch.min(ratio*adv, clipped_ratio*adv).mean()")

check()

## 7. Train

**Nothing here is marked 🎯: you do not write anything.** Read the shape, run the
cell, watch the number climb. The loop below is ordinary PPO, and it calls the
`ppo_losses` you just wrote.

It plays **exactly the same 102,400 steps** REINFORCE played, with the same
network, the same discount and the same entropy bonus. REINFORCE keeps the larger
learning rate that suits it (`1e-3` against PPO's `3e-4`); apart from that, what
differs is the objective, and what the objective lets you do with the data.

In pseudocode, the whole cell is three phases repeated fifty times:

```
repeat 50 times:                      # one "update"

  1. PLAY
     play 2048 steps with the CURRENT actor, and for each step write down:
        state, action, log-probability of that action, the critic's value,
        the reward, and whether the life ended

  2. SCORE
     walk BACKWARDS through those 2048 steps:
        surprise  = reward + discount x value(next state) - value(this state)
        advantage = surprise + discount x lambda x advantage(next step)

  3. IMPROVE
     repeat 10 times:                 # ten passes over the SAME 2048 steps
        for each minibatch of 64:
           loss = YOUR THREE LINES                  # moves the ACTOR
                + 0.5 x (critic's error)^2          # teaches the CRITIC
                - 0.01 x (actor's entropy)          # pays the actor to stay undecided
           nudge both networks downhill

  throw the 2048 steps away: the actor has moved, so they are stale
```

One optimiser holds both networks' parameters, which is why a single
`loss.backward()` and a single `opt.step()` improve the two of them at once.

That `repeat 10 times` is the reason PPO exists, and it is exactly what REINFORCE
could not do. Without the probability ratio you get **one** pass over the data
before it goes stale, which is the single gradient step you watched REINFORCE
take. Your three lines buy the other nine passes, and 32 minibatches inside each:
320 gradient steps per rollout instead of 1, and 16,000 over the run instead of
50, off the very same 102,400 steps of the game.

Notice too that step 1 is the loop you wrote in Task 1, with `play_one_life`'s
coin flip replaced by the actor, and the bookkeeping kept instead of thrown away.

**About two minutes** (measured: 117 seconds on a Colab CPU runtime). Watch the
mean return climb past where REINFORCE stalled.

It comes in the same three pieces as REINFORCE, and the first two are worth
comparing against their counterparts: `advantages` against `returns_to_go`, and
`collect_steps` against `collect_lives`. Those two differences **are** the
difference between the algorithms.

#### First, the scoring

`returns_to_go` measured every action against the whole rest of its life. This
measures each step against what the critic expected instead. The **surprise** at
step $t$ is how much better the reward plus the next state's value turned out
than the value of the state you were in:

$$\delta_t \;=\; r_t + \gamma\, V(s_{t+1}) - V(s_t)$$

Then those surprises are discounted backwards exactly as the rewards were, which
is the only line this shares with `returns_to_go`.

In [ ]:
UPDATES, ROLLOUT, EPOCHS, MINIBATCH = 50, 2048, 10, 64
LAM, LR, VF_COEF = 0.95, 3e-4, 0.5          # GAMMA and ENT_COEF are REINFORCE's

def advantages(R, V, D, last_v, gamma=GAMMA, lam=LAM):
    '''How much better each action was than its state deserved.'''
    n = len(R)
    adv, run = torch.zeros(n), 0.0
    for t in reversed(range(n)):
        nextv   = last_v if t == n - 1 else V[t + 1]
        nonterm = 1.0 - D[t]                              # 0 if the life ended here
        surprise = R[t] + gamma * nextv * nonterm - V[t]
        run = surprise + gamma * lam * nonterm * run      # same backwards walk
        adv[t] = run
    return adv

#### Then the playing, with one difference that matters

`collect_lives` waited for each life to finish. This one stops after exactly 2048
steps, **even if the bird is still flying**, and that is why the two extra things
below exist:

- `state` carries the half-finished life over to the next update, so the game
  goes on from where it stopped rather than restarting.
- the critic is asked for a value at every step, because when a rollout ends
  mid-flight something has to say what the unfinished future is worth.

It also writes down `LOGP`, the log-probability the policy gave each action **at
the time it was taken**. That is the $\pi_{\theta_{old}}$ in your ratio.

In [ ]:
def collect_steps(actor, critic, env, state, n_steps):
    '''Play exactly n_steps, finished lives or not.'''
    obs, ep_ret = state
    O = torch.zeros(n_steps, obs.shape[0]); A = torch.zeros(n_steps, dtype=torch.long)
    LOGP = torch.zeros(n_steps); R = torch.zeros(n_steps)
    D = torch.zeros(n_steps);    V = torch.zeros(n_steps)
    finished = []

    for t in range(n_steps):
        with torch.no_grad():
            d = actor.dist(obs); a = d.sample()
            O[t], A[t], LOGP[t], V[t] = obs, a, d.log_prob(a), critic(obs)
        nobs, r, term, trunc, _ = env.step(int(a))
        R[t], D[t] = float(r), float(term or trunc)
        ep_ret += float(r)
        if term or trunc:
            finished.append(ep_ret); ep_ret = 0.0; nobs, _ = env.reset()
        obs = torch.as_tensor(nobs, dtype=torch.float32)

    return (O, A, LOGP, R, D, V), (obs, ep_ret), finished

#### And the loop that uses them

Same three phases as REINFORCE. The only structural difference is inside phase 3,
where the batch is walked over ten times in minibatches of 64 instead of once,
and that is what your three lines paid for.

In [ ]:
def train(seed=SEED, updates=UPDATES):
    torch.manual_seed(seed); np.random.seed(seed)     # re-seed so re-running is repeatable
    env = make_env()
    actor  = Actor(env.observation_space.shape[0], env.action_space.n)
    critic = Critic(env.observation_space.shape[0])
    params = [*actor.parameters(), *critic.parameters()]
    opt = torch.optim.Adam(params, lr=LR)

    obs, _ = env.reset(seed=seed)
    state = (torch.as_tensor(obs, dtype=torch.float32), 0.0)
    log, curve, t0 = [], [], time.time()

    for update in range(updates):
        batch, state, finished = collect_steps(actor, critic, env, state, ROLLOUT)
        O, A, LOGP, R, D, V = batch                             # 1. PLAY
        log += finished

        with torch.no_grad(): last_v = critic(state[0])         # 2. SCORE
        adv = advantages(R, V, D, last_v)
        ret = adv + V

        idx = np.arange(ROLLOUT)                                # 3. IMPROVE, 320 times
        for _ in range(EPOCHS):
            np.random.shuffle(idx)
            for s in range(0, ROLLOUT, MINIBATCH):
                mb = idx[s:s + MINIBATCH]
                d = actor.dist(O[mb])
                a_mb = adv[mb]; a_mb = (a_mb - a_mb.mean()) / (a_mb.std() + 1e-8)

                _, policy_loss = ppo_losses(d.log_prob(A[mb]), LOGP[mb], a_mb)   # <<< your code

                value_loss = ((critic(O[mb]) - ret[mb]) ** 2).mean()
                loss = policy_loss + VF_COEF * value_loss - ENT_COEF * d.entropy().mean()
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(params, 0.5)
                opt.step()

        curve.append(float(np.mean(log[-20:])) if log else 0.0)
        if (update + 1) % 5 == 0:
            print("update %2d/%d   steps %6d   mean return %6.2f   %4.0fs"
                  % (update + 1, updates, (update + 1) * ROLLOUT, curve[-1], time.time() - t0), flush=True)
    env.close()
    return actor, critic, curve

trained_actor, trained_critic, curve = train()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3.2))
plt.plot(rf_xs, rf_curve, label="REINFORCE")
plt.plot(np.arange(1, len(curve) + 1) * ROLLOUT, curve, label="PPO")
plt.xlabel("steps of the game played"); plt.ylabel("mean return, last 20 lives")
plt.title("the same 102,400 steps, two objectives")
plt.legend(); plt.grid(alpha=.3); plt.show()

## 8. Watch all three

The clips below are recorded with each actor's **best** action at every step
rather than a sampled one, so what you see is what was actually learned and not a
lucky or unlucky draw. The critic is not consulted here: it existed to judge the
actor's actions during training, not to play.

In [ ]:
frames_after, ret_after, pipes_after = rollout(trained_actor, seed=SEED)
for name, f, p, r in [("untrained", frames_before, pipes_before, ret_before),
                      ("REINFORCE", frames_rf,     pipes_rf,     ret_rf),
                      ("PPO      ", frames_after,  pipes_after,  ret_after)]:
    print("%s: survived %3d frames (%4.1f seconds), %d pipes, return %6.2f"
          % (name, len(f), len(f) / 30, p, r))

after_path = save_video(frames_after, "after.mp4")
play(show_video(before_path, 210, "untrained: %d pipes, return %.2f" % (pipes_before, ret_before)),
     show_video(rf_path,     210, "REINFORCE: %d pipes, return %.2f" % (pipes_rf, ret_rf)),
     show_video(after_path,  210, "PPO: %d pipes, return %.2f"       % (pipes_after, ret_after)))

## What to notice

**The reward never told it how to fly.** It only ever scored the outcome: a
little for surviving a frame, more for a pipe, a penalty for dying or for
drifting off the top. Nobody wrote a rule about when to flap. The flying came
out of trying and keeping what worked.

**Same game, same network, same number of steps.** The gap between the second
clip and the third is not more data or a bigger model. It is what each objective
does with the data it has: REINFORCE scores every action by how the whole life
turned out and then uses the batch once; PPO scores each action against what the
critic thought the state was worth, and reuses the batch 320 times.

**The loop never changed.** The one you wrote in Task 1 is the same loop under
both algorithms. All that happened is that the coin flip became the actor, and
the numbers it threw away got written down and learned from.

**The learning curve is bumpy.** That is normal and it is the subject of the
lecture: each update is measured from a noisy sample, and PPO's clipping is what
stops one bad sample throwing the policy away.

**It is not finished.** Two minutes of training on a laptop-sized machine buys
a competent beginner, not an expert. The published agents that play forever train
for tens of millions of steps.

### If you have time

- Give REINFORCE six times the budget: `train_reinforce(total_steps=600_000)`.
  It stays where it is. The rut is the algorithm, not the patience.
- Set `CLIP_EPS = 1000.0` (effectively no clipping) and train again. This is the
  experiment the lecture describes: watch the run become unstable.
- Set `ENT_COEF = 0.0`. That removes the reward for staying undecided. The policy
  often collapses onto one action early and then stops improving.
- Change `SEED` and re-run. The agent is not identical every time.